# **Packages, Please**

## **Relating: packages analysis**

A mail delivery system in the city of Boston is simulated. As the person responsible for package tracking, the objective is to investigate cases of lost packages reported by customers.

For each case, the goal is to determine:

- The current location of the package
- The type of location (residential, business, etc.)
- The contents of the package

---

## **Database Structure**

The `packages.db` database allows tracking the movement of packages throughout the city using multiple related tables.

### **`addresses`**

Contains the addresses registered in the system.

- `id`: unique address identifier
- `address`: physical address
- `type`: type of location (residential, commercial, etc.)

### **`drivers`**

Contains information about the drivers responsible for deliveries.

- `id`: driver identifier
- `name`: driver name

### **`packages`**

Contains general information about the packages.

- `id`: package identifier
- `contents`: package contents
- `from_address_id`: origin address
- `to_address_id`: destination address (not necessarily the final location)

### **`scans`**

Stores the package tracking history.

- `id`: scan record identifier
- `driver_id`: driver who performed the scan
- `package_id`: scanned package
- `address_id`: scan location
- `action`: action performed (`"Pick"` or `"Drop"`)
- `timestamp`: date and time of the event

---

## **Relationships Between Entities**

- A `driver` can perform multiple `scans`, but each `scan` is performed by only one `driver`.

- Each `scan` is associated with a single `package`, while a `package` can have multiple `scans` throughout its journey.

- Each `scan` occurs at one `address`, while multiple `scans` can occur at the same `address`.

- Each `package` has an origin address (`from_address_id`) and a destination address (`to_address_id`), both referencing the `addresses` entity.

These relationships make it possible to reconstruct the complete journey of a package from its origin to its current location.

In [1]:
import pandas as pd 
import sqlite3
conn = sqlite3.connect("packages.db")

def show_table(name):
    query = f"SELECT*FROM {name} LIMIT 5"
    return pd.read_sql_query(query,conn) 

In [16]:
show_table("addresses")

,id,address,type
0,1,7660 Sharon Street,Residential
1,2,60 Drake Place,Residential
2,3,88 City Point Court,Residential
3,4,266 Dorchester Avenue,Residential
4,5,2 Otis Place,Residential


In [17]:
show_table("drivers")

,id,name
0,1,Matthew
1,2,Isabel
2,3,Julianna
3,4,Varsha
4,5,Jacob


In [18]:
show_table("packages")

,id,contents,from_address_id,to_address_id
0,1,Sticky notes,8575,4635
1,2,Shears,8431,7641
2,3,Index cards,2107,3624
3,4,Glue stick,6954,3740
4,5,Trading cards,4530,5564


In [19]:
show_table("scans")

,id,driver_id,package_id,address_id,action,timestamp
0,1,11,8502,1063,Pick,2023-07-11 15:16:05.340221
1,2,8,3320,9551,Pick,2023-07-11 15:19:39.359315
2,3,1,2879,9589,Pick,2023-07-11 15:23:47.208594
3,4,20,2240,3464,Pick,2023-07-11 15:28:16.920400
4,5,1,8664,8013,Pick,2023-07-11 15:32:42.265139


## **Analysis Cases**

### **Case 1: Lost Letter**

The shipment of a congratulatory letter sent from `900 Somerville Avenue` to `2 Finnegan Street` is analyzed.

The objective is to determine whether the letter was successfully delivered and identify its current status within the system.

### **Objective**

- Determine the type of residence where the letter was delivered

### **Approach:**

**1.** Obtain the id of the origin address

**2.** Obtain the id of the associated package
    
    - There may be multiple packages, so filtering is done by contents

**3.** Obtain the latest scan of the package (ORDER BY timestamp DESC)

**4.** Obtain the type of the final address

In [35]:
query = """

SELECT "type" FROM "addresses" WHERE "id" = (
  SELECT "address_id" FROM "scans" WHERE "package_id" = (
    SELECT "id" FROM "packages" WHERE "from_address_id" = (
      SELECT "id" FROM "addresses" WHERE "address" = '900 Somerville Avenue')
      AND "contents" LIKE '%ongratulatory%'
  )
  ORDER BY "timestamp" DESC LIMIT 1
);

"""

pd.read_sql_query(query,conn)

,type
0,Residential


### **Objective**

- Determine the address where it was delivered

In [37]:
query = """

SELECT "address" FROM "addresses" WHERE "id" = (
  SELECT "address_id" FROM "scans" WHERE "package_id" = (
    SELECT "id" FROM "packages" WHERE "from_address_id" = (
      SELECT "id" FROM "addresses" WHERE "address" = '900 Somerville Avenue')
      AND "contents" LIKE '%ongratulatory%'
  )
  ORDER BY "timestamp" DESC LIMIT 1
);

"""

pd.read_sql_query(query,conn)

,address
0,2 Finnigan Street


### **Case 2: Delivery Without Sender**

An investigation is conducted on a package with no registered origin address, whose contents correspond to a recreational object (a bath toy).

### **Objective**

- Determine the type of residence where the package was delivered

### **Approach**

**1.** Identify package without origin address (from_address_id IS NULL)

**2.** There may be multiple packages without an address, filter by contents

**3.** Get the latest scan of the package

**4.** Get the address type

In [3]:
query = """ 

SELECT "type" FROM "addresses" WHERE "id" = (
  SELECT "address_id" FROM "scans" WHERE "package_id" = (
    SELECT "id" FROM "packages" 
    WHERE "from_address_id" IS NULL 
    AND "contents" LIKE '%uck%'
  )
  ORDER BY "timestamp" DESC LIMIT 1
);

"""

pd.read_sql_query(query,conn)

,type
0,Police Station


### **Objective**

- Determine the contents of the package

In [5]:
query = """ 

SELECT "contents" FROM "packages" 
WHERE "from_address_id" IS NULL 
AND "contents" LIKE '%uck%';

""" 

pd.read_sql_query(query,conn)

,contents
0,Duck debugger



### **Case 3: Undelivered Gift**

A package sent from `109 Tileston Street` to `728 Maple Place` is analyzed due to delivery delays.

The objective is to track the package and determine its current status within the delivery system.

### **Objective**

- Determine the contents of the gift

### **Approach**

**1.** Filter by origin address

**2.** There may be multiple packages sent from the same origin, filter by destination address

In [7]:
query = """ 

SELECT "contents" FROM "packages" WHERE "from_address_id" = (
  SELECT "id" FROM "addresses" WHERE "address" = '109 Tileston Street'
)
AND "to_address_id" = (
  SELECT "id" FROM "addresses" WHERE "address" = '728 Maple Place'
);

"""

pd.read_sql_query(query,conn)

,contents
0,Flowers


### **Objective**

- Determine who received it

### **Approach**

**1.** Identify the package with specific origin and destination

**2.** Get the driver_id associated with the latest scan of the package (timestamp DESC)

**3.** Identify the associated driver

In [8]:
query = """ 

SELECT "name" FROM "drivers" WHERE "id" = (
  SELECT "driver_id" FROM "scans" WHERE "package_id" = (
    SELECT "id" FROM "packages" WHERE "from_address_id" = (
      SELECT "id" FROM "addresses" WHERE "address" = '109 Tileston Street'
    )
    AND "to_address_id" = (
      SELECT "id" FROM "addresses" WHERE "address" = '728 Maple Place'
    )
  )
  ORDER BY "timestamp" DESC LIMIT 1
);

"""

pd.read_sql_query(query,conn)

,name
0,Mikel
